In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import pandas as pd

Path to ROCo dataset files

In [3]:
train_images_path = "/content/drive/MyDrive/ROCo/radiology/img"
train_captions_path = "/content/drive/MyDrive/ROCo/radiology/traindata.csv"
val_images_path = "/content/drive/MyDrive/ROCo/validation/radiology/img"
val_captions_path = "/content/drive/MyDrive/ROCo/validation/radiology/valdata.csv"

Loading and Preprocessing of data

In [4]:
def load_captions(captions_path):
    df = pd.read_csv(captions_path)
    return df

In [5]:
from keras.applications import VGG19
from keras.applications.vgg19 import preprocess_input
from keras.preprocessing.image import load_img, img_to_array
from keras.models import Model

In [6]:
def preprocess_images(images_path):
    vgg19 = VGG19(weights="imagenet")
    feature_extractor = Model(inputs=vgg19.input, outputs=vgg19.get_layer("fc2").output)

    features = {}
    for img_name in os.listdir(images_path):
        img_path = os.path.join(images_path, img_name)
        img = load_img(img_path, target_size=(224, 224))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)

        feature = feature_extractor.predict(img_array)
        features[img_name] = feature
    return features

Feature extraction for images

In [7]:
train_features = preprocess_images(train_images_path)
val_features = preprocess_images(val_images_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 755ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 730ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 735ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 734ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 723ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 725ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 736ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 752ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 751ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 734ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 732ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 892ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 734ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 742ms/step
1/1 ━━━━━━━━━━━━

Loading Captions

In [8]:
train_captions = load_captions(train_captions_path)
val_captions = load_captions(val_captions_path)

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from sklearn.model_selection import train_test_split
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_captions['caption'])

In [10]:
vocab_size = len(tokenizer.word_index) + 1
max_len = max([len(c.split()) for c in train_captions['caption']])

In [11]:
def create_sequences(features, captions, tokenizer, max_len):
    X1, X2, y = [], [], []
    missing_keys = []
    valid_keys = set(features.keys())

    for label, caption in zip(captions['name'], captions['caption']):
        if label in valid_keys:
            seq = tokenizer.texts_to_sequences([caption])[0]
            if len(seq) < 2:
                print(f"Skipping caption '{caption}' for label '{label}' as it is too short.")
                continue

            for i in range(1, len(seq)):
                in_seq, out_seq = seq[:i], seq[i]
                in_seq = pad_sequences([in_seq], maxlen=max_len)[0]
                out_seq_one_hot = np.zeros(vocab_size)
                out_seq_one_hot[out_seq] = 1

                feature = features[label].flatten() if len(features[label].shape) > 1 else features[label]
                X1.append(feature)
                X2.append(in_seq)
                y.append(out_seq_one_hot)
        else:
            missing_keys.append(label)

    if missing_keys:
        print(f"Number of missing keys: {len(missing_keys)}")
        print(f"Missing keys: {missing_keys[:10]}{'...' if len(missing_keys) > 10 else ''}")  # Print first 10 missing keys

    return np.array(X1), np.array(X2), np.array(y)


In [12]:
X1_train, X2_train, y_train = create_sequences(train_features, train_captions, tokenizer, max_len)
print(f"X1_train shape: {X1_train.shape}")
print(f"X2_train shape: {X2_train.shape}")
print(f"y_train shape: {y_train.shape}")


Number of missing keys: 65407
Missing keys: ['PMC4083729_AMHSR-4-14-g002.jpg', 'PMC2837471_IJD2009-150251.001.jpg', 'PMC2505281_11999_2007_30_Fig6_HTML.jpg', 'PMC3745845_IJD2013-683423.005.jpg', 'PMC4917066_amjcaserep-17-301-g001.jpg', 'PMC4805615_13244_2016_481_Fig12_HTML.jpg', 'PMC2584650_1757-1626-1-193-1.jpg', 'PMC3283944_JISP-15-414-g006.jpg', 'PMC4946383_HI-10-1-25-g003.jpg', 'PMC5646151_TOORTHJ-11-882_F2.jpg']...
X1_train shape: (1294, 4096)
X2_train shape: (1294, 410)
y_train shape: (1294, 37660)


In [13]:
print(f"Number of valid samples: {len(X1_train)}")

Number of valid samples: 1294


In [14]:
X1_train, X2_train, y_train = create_sequences(train_features, train_captions, tokenizer, max_len)
X1_val, X2_val, y_val = create_sequences(val_features, val_captions, tokenizer, max_len)

Number of missing keys: 65407
Missing keys: ['PMC4083729_AMHSR-4-14-g002.jpg', 'PMC2837471_IJD2009-150251.001.jpg', 'PMC2505281_11999_2007_30_Fig6_HTML.jpg', 'PMC3745845_IJD2013-683423.005.jpg', 'PMC4917066_amjcaserep-17-301-g001.jpg', 'PMC4805615_13244_2016_481_Fig12_HTML.jpg', 'PMC2584650_1757-1626-1-193-1.jpg', 'PMC3283944_JISP-15-414-g006.jpg', 'PMC4946383_HI-10-1-25-g003.jpg', 'PMC5646151_TOORTHJ-11-882_F2.jpg']...
Number of missing keys: 8130
Missing keys: ['PMC3970251_CRIONM2014-931546.003.jpg', 'PMC2766744_cios-1-176-g005.jpg', 'PMC3789931_poljradiol-78-3-35-g001.jpg', 'PMC2676075_p147_fig4a.jpg', 'PMC5292123_CRIGM2017-1710501.002.jpg', 'PMC4756892_CMJ-128-2946-g004.jpg', 'PMC2494540_1757-1626-1-52-1.jpg', 'PMC3339065_NAJMS-2-392-g001.jpg', 'PMC2636159_IndianJOphthalmol-56-269-g003.jpg', 'PMC5625558_1657-9534-cm-48-02-00088-gf1.jpg']...


In [15]:
from keras.layers import Input, LSTM, Embedding, Dense, Dropout, RNN, SimpleRNN
from keras.models import Model
def build_lstm_model(vocab_size, max_len):
    inputs1 = Input(shape=(4096,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    inputs2 = Input(shape=(max_len,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = LSTM(256)(se1)

    decoder1 = Dense(256, activation='relu')((fe2))
    outputs = Dense(vocab_size, activation='softmax')(decoder1)
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

In [16]:
def build_rnn_model(vocab_size, max_len):
    inputs1 = Input(shape=(4096,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    inputs2 = Input(shape=(max_len,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = SimpleRNN(256)(se1)

    decoder1 = Dense(256, activation='relu')((fe2))
    outputs = Dense(vocab_size, activation='softmax')(decoder1)
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

In [17]:
lstm_model = build_lstm_model(vocab_size, max_len)
lstm_model.fit([X1_train, X2_train], y_train, epochs=20, batch_size=64, validation_data=([X1_val, X2_val], y_val))

Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 400ms/step - loss: 9.2939 - val_loss: 8.5759
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 308ms/step - loss: 5.7425 - val_loss: 8.5685
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 288ms/step - loss: 4.7596 - val_loss: 8.5659
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 333ms/step - loss: 4.1242 - val_loss: 8.5228
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 300ms/step - loss: 3.8345 - val_loss: 8.6053
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 296ms/step - loss: 3.6348 - val_loss: 8.5096
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 326ms/step - loss: 3.5778 - val_loss: 8.7284
Epoch 8/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 292ms/step - loss: 3.5409 - val_loss: 8.5016
Epoch 9/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 282ms/step - loss: 3.5374 - val_loss: 8.5532
Epoch 10/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 373ms/step - loss: 3.4670 - val_loss: 8.6334
Epoch 11/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 285ms/step - loss: 3.4171 - val_loss: 8.5209
Epoch 12/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 390m

In [18]:
rnn_model = build_rnn_model(vocab_size, max_len)
rnn_model.fit([X1_train, X2_train], y_train, epochs=20, batch_size=64, validation_data=([X1_val, X2_val], y_val))

Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 346ms/step - loss: 9.2027 - val_loss: 8.3197
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 383ms/step - loss: 5.8370 - val_loss: 8.6320
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 302ms/step - loss: 4.8042 - val_loss: 8.6872
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 301ms/step - loss: 4.1105 - val_loss: 8.6605
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 375ms/step - loss: 3.8067 - val_loss: 8.7447
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 317ms/step - loss: 3.7055 - val_loss: 8.6439
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 302ms/step - loss: 3.6054 - val_loss: 8.5313
Epoch 8/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 369ms/step - loss: 3.5050 - val_loss: 8.5335
Epoch 9/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 392ms/step - loss: 3.4842 - val_loss: 8.4606
Epoch 10/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 304ms/step - loss: 3.4818 - val_loss: 8.6269
Epoch 11/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 343ms/step - loss: 3.4300 - val_loss: 8.4314
Epoch 12/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 419

In [19]:
lstm_loss = lstm_model.evaluate([X1_val, X2_val], y_val)
rnn_loss = rnn_model.evaluate([X1_val, X2_val], y_val)

31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 8.7213
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 8.4035


In [20]:
print(f"LSTM Loss: {lstm_loss}")
print(f"RNN Loss: {rnn_loss}")

LSTM Loss: 8.798356056213379
RNN Loss: 8.431005477905273


Hyperparameter Tuning

RandomizedSearch

In [21]:
import numpy as np
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_classes=2, random_state=42)

from scipy.stats import randint
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
	"max_depth": [3, None],
	"max_features": randint(1, 9),
	"min_samples_leaf": randint(1, 9),
	"criterion": ["gini", "entropy"]
}

tree = DecisionTreeClassifier()
tree_cv = RandomizedSearchCV(tree, param_dist, cv=5)
tree_cv.fit(X, y)

print("Tuned Decision Tree Parameters: {}".format(tree_cv.best_params_))
print("Best score is {}".format(tree_cv.best_score_))

Tuned Decision Tree Parameters: {'criterion': 'entropy', 'max_depth': None, 'max_features': 7, 'min_samples_leaf': 5}
Best score is 0.851


Gridsearch

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import numpy as np
from sklearn.datasets import make_classification


X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_classes=2, random_state=42)

c_space = np.logspace(-5, 8, 15)
param_grid = {'C': c_space}

logreg = LogisticRegression()
logreg_cv = GridSearchCV(logreg, param_grid, cv=5)

logreg_cv.fit(X, y)

print("Tuned Logistic Regression Parameters: {}".format(logreg_cv.best_params_))
print("Best score is {}".format(logreg_cv.best_score_))

Tuned Logistic Regression Parameters: {'C': 0.006105402296585327}
Best score is 0.853


Hyperopt

In [23]:
!pip install hyperopt

In [26]:
from hyperopt import fmin, tpe, hp, anneal, Trials, STATUS_OK

In [27]:
def objective(params):
    C = params['C']
    logreg = LogisticRegression(C=C)
    logreg.fit(X, y)
    accuracy = logreg.score(X, y)
    return {'loss': -accuracy, 'status': STATUS_OK}

space = {
    'C': hp.loguniform('C', -5, 8)
}

trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials
)

print("Best hyperparameters:", best)
best_trial = trials.best_trial

print("Best loss:", best_trial['result']['loss'])
print("Best C:", best_trial['misc']['vals']['C'][0])

100%|██████████| 50/50 [00:00<00:00, 76.33trial/s, best loss: -0.86]
Best hyperparameters: {'C': 0.02372618868130962}
Best loss: -0.86
Best C: 0.02372618868130962


Evaluation Metrics

In [28]:
y_pred = lstm_model.predict([X1_val, X2_val])
y_pred_classes = np.argmax(y_pred, axis=-1)
y_val_classes = np.argmax(y_val, axis=-1)

31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step


In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [31]:
accuracy = accuracy_score(y_val_classes, y_pred_classes)
precision = precision_score(y_val_classes, y_pred_classes, average='weighted')
recall = recall_score(y_val_classes, y_pred_classes, average='weighted')
f1 = f1_score(y_val_classes, y_pred_classes, average='weighted')
roc_auc = roc_auc_score(y_val, y_pred, multi_class='ovr')

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_ranking.py:375: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_ranking.py:375: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  w

In [32]:
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("F1-Score: ", f1)
print("AUC-ROC: ", roc_auc)

Accuracy:  0.04485219164118247
Precision:  0.005954377370219411
Recall:  0.04485219164118247
F1-Score:  0.01010849479681205
AUC-ROC:  nan


In [33]:
from sklearn.metrics import classification_report
unique_classes = np.unique(y_val_classes)
target_names = [str(cls) for cls in unique_classes]
report = classification_report(y_val_classes, y_pred_classes, target_names=target_names, labels=unique_classes)
print("Classification Report: ")
print(report)

Classification Report: 
              precision    recall  f1-score   support

           1       0.06      0.56      0.11        64
           2       0.03      0.13      0.05        30
           3       0.00      0.00      0.00        25
           4       0.00      0.00      0.00        22
           5       0.03      0.04      0.03        26
           6       0.00      0.00      0.00        14
           7       0.00      0.00      0.00         9
           8       0.00      0.00      0.00        13
           9       0.00      0.00      0.00        13
          10       0.00      0.00      0.00        14
          11       0.00      0.00      0.00        10
          12       0.00      0.00      0.00        14
          13       0.00      0.00      0.00        10
          14       0.00      0.00      0.00         9
          15       0.00      0.00      0.00         6
          16       0.00      0.00      0.00        17
          17       0.00      0.00      0.00         4
   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=5c1d50935873e4a96b3ed3b6afeecf88432f91b4e16988377a869412b695b7a3
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [36]:
import evaluate
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

reference = ["Computed tomography scan in axial view showing"]
candidate = ["Computed tomography scan in axial view showing"]

bleu_results = bleu_metric.compute(predictions=candidate, references=reference)
print(f"BLEU Score: {bleu_results['bleu'] * 100:.2f}")

rouge_results = rouge_metric.compute(predictions=candidate, references=reference)

print(f"ROUGE-1 F1 Score: {rouge_results['rouge1']:.2f}")
print(f"ROUGE-L F1 Score: {rouge_results['rougeL']:.2f}")

BLEU Score: 100.00
ROUGE-1 F1 Score: 1.00
ROUGE-L F1 Score: 1.00
